# AdaptiveDepth Phase 6 - compact router dataset
This application-track notebook compares the corrected full model with `route_target_21` on held-out prompts. It produces prompt features, route-regret labels, latency measurements, and a Phase 7 readiness gate. The full model remains the fallback. Use a T4 GPU runtime.

In [ ]:
import base64, json, os, pathlib, shutil, subprocess, sys, zipfile

REPO_URL = 'https://github.com/VedantPandhare/AdaptiveDepth-Dynamic-Layer-Routing-for-LLMs.git'
base = pathlib.Path('/content')
repo = base / 'AdaptiveDepth-Dynamic-Layer-Routing-for-LLMs'

def read_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

if not repo.exists():
    clone_env = os.environ.copy()
    github_token = read_secret('GITHUB_TOKEN')
    if github_token:
        credential = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
        clone_env.update({'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.extraHeader', 'GIT_CONFIG_VALUE_0': f'Authorization: Basic {credential}'})
    clone = subprocess.run(['git', 'clone', REPO_URL, str(repo)], env=clone_env, capture_output=True, text=True)
    if clone.returncode:
        detail = clone.stderr.strip().splitlines()[-1] if clone.stderr.strip() else 'unknown Git error'
        raise RuntimeError(f'Git clone failed: {detail}\nFor a private repository, add a read-only GITHUB_TOKEN to Colab Secrets.')
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[data]'], check=True)
subprocess.run(['nvidia-smi'], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before running Phase 6'
print('Repository:', repo)
print('GPU:', torch.cuda.get_device_name(0))

## Mount Drive and stage the corrected model
This uses `MyDrive/AdaptiveDepth-results/phase5_corrected_student.zip`. The original step-500 model is not needed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
drive_results = pathlib.Path('/content/drive/MyDrive/AdaptiveDepth-results')
archive = drive_results / 'phase5_corrected_student.zip'
extract_root = pathlib.Path('/content/phase5_corrected_for_phase6')
if not archive.is_file():
    raise FileNotFoundError(f'Missing corrected checkpoint archive: {archive}')
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True)
with zipfile.ZipFile(archive) as bundle:
    bundle.extractall(extract_root)
matches = [path.parent for path in extract_root.rglob('model.safetensors')]
if len(matches) != 1:
    raise RuntimeError(f'Expected one model checkpoint in the archive, found: {matches}')
checkpoint = matches[0]
required = ['model.safetensors', 'config.json', 'tokenizer.json', 'tokenizer_config.json']
missing = [name for name in required if not (checkpoint / name).is_file()]
if missing:
    raise RuntimeError(f'Incomplete corrected checkpoint: {missing}')
print('Corrected checkpoint:', checkpoint)
print('Model size GB:', round((checkpoint / 'model.safetensors').stat().st_size / 1024**3, 3))

## Choose smoke or full collection
Run smoke first. After it completes, set `RUN_FULL = True` and rerun this cell and the final two cells. A smoke `revise_phase6` verdict is diagnostic and does not block the full run.

In [ ]:
RUN_FULL = False
if RUN_FULL:
    dataset = pathlib.Path('data/phase6_calibration.jsonl')
    if not dataset.exists():
        subprocess.run([
            'adaptivedepth', 'prepare-phase6-data',
            '--output', str(dataset), '--max-examples', '200',
            '--skip-examples', '2300', '--seed', '17',
        ], check=True)
    config = 'configs/phase6_t4.yaml'
    workspace = pathlib.Path('experiments/phase6')
else:
    dataset = pathlib.Path('data/phase6_calibration_smoke.jsonl')
    config = 'configs/phase6_smoke.yaml'
    workspace = pathlib.Path('experiments/phase6_smoke')
print('Mode:', 'FULL' if RUN_FULL else 'SMOKE')
print('Dataset:', dataset, 'examples:', sum(1 for _ in dataset.open(encoding='utf-8')))
print('Workspace:', workspace)

In [ ]:
subprocess.run([
    'adaptivedepth', 'run-phase6',
    '--checkpoint', str(checkpoint),
    '--config', config,
    '--route-file', 'configs/phase6_route.yaml',
    '--dataset', str(dataset),
    '--workspace', str(workspace),
    '--resume',
], check=True)

In [ ]:
print((workspace / 'gate.md').read_text())
output_name = 'phase6_artifacts.zip' if RUN_FULL else 'phase6_smoke_artifacts.zip'
output_zip = pathlib.Path('/content') / output_name
with zipfile.ZipFile(output_zip, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in workspace.rglob('*'):
        if path.is_file():
            bundle.write(path, workspace.name + '/' + str(path.relative_to(workspace)))
drive_results.mkdir(parents=True, exist_ok=True)
shutil.copy2(output_zip, drive_results / output_zip.name)
shutil.copy2(workspace / 'gate.md', drive_results / ('phase6_gate.md' if RUN_FULL else 'phase6_smoke_gate.md'))
shutil.copy2(workspace / 'gate.json', drive_results / ('phase6_gate.json' if RUN_FULL else 'phase6_smoke_gate.json'))
print('Saved:', drive_results / output_zip.name)
if not RUN_FULL:
    print('Smoke collection complete. Set RUN_FULL = True and rerun the final three code cells.')
else:
    print('Upload phase6_artifacts.zip or paste the gate output for analysis.')